# 🦈 Shark Video Annotation Tool 

**Everything in one place:**
1. Upload video
2. Do first annotation in Rovoflow
3. SAM 2 tracks automatically
4. Compare original vs tracking
5. Export annotations

---

## 1: SAM 2 Model Initialization

#### Run in every new session. Loads SAM 2 model into GPU memory.

In [2]:
import sys
print(sys.executable)
!which pip
!pip --version

/Users/kamronaggor/Desktop/School/URI/AI Lab/OceanDetect/sam2-vidio-annotation/.venv/bin/python
/Users/kamronaggor/Desktop/School/URI/AI Lab/OceanDetect/sam2-vidio-annotation/.venv/bin/pip
pip 25.1.1 from /Users/kamronaggor/Desktop/School/URI/AI Lab/OceanDetect/sam2-vidio-annotation/.venv/lib/python3.11/site-packages/pip (python 3.11)


In [11]:
# ============================================
# SAM 2 MODEL INITIALIZATION
# ============================================
# This cell:
# - Loads SAM 2 model into GPU
# - Sets up deterministic behavior for consistent results across different nodes
# - Clears any cached modules from previous sessions

import os
import pathlib as path
import sys
import torch
import numpy as np
import random
import gc

# Move to SAM2 directory (required for import)
sam2_dir = f'{path.Path.cwd()}/sam2'
os.chdir(os.path.expanduser(sam2_dir))
print(f"Working directory: {os.getcwd()}")

# Add to path
if sam2_dir not in sys.path:
    sys.path.insert(0, sam2_dir)

# Clear cached SAM2 modules
modules_to_clear = [k for k in list(sys.modules.keys()) if 'sam2' in k.lower()]
for mod in modules_to_clear:
    del sys.modules[mod]
gc.collect()
print(f"Cleared {len(modules_to_clear)} cached modules")

# GPU cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    gc.collect()

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Configure PyTorch for consistent results across different GPUs
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False  # Critical for A100/L40S/H100 consistency

# Set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ Using CPU')

# Import SAM 2
try:
    from sam2.build_sam import build_sam2_video_predictor
    print('✓ SAM 2 imported successfully')
except Exception as e:
    print(f'❌ SAM 2 import failed: {e}')
    raise

# Load model checkpoint
checkpoint_path = '/home/zihara_delgado_uri_edu/checkpoints/sam2_hiera_large.pt'
config_path = 'sam2_hiera_l.yaml'

print(f'Loading checkpoint: {checkpoint_path}')

try:
    predictor = build_sam2_video_predictor(
        config_path,
        checkpoint_path,
        device=device
    )
    print('✓ Model loaded successfully')
except Exception as e:
    print(f'❌ Model loading failed: {e}')
    raise

# Warmup (stabilizes first predictions)
try:
    print('Warming up model...')
    dummy_input = torch.randn(1, 3, 1024, 1024).to(device)
    with torch.no_grad():
        _ = predictor.image_encoder(dummy_input)
    torch.cuda.synchronize()
    del dummy_input
    torch.cuda.empty_cache()
    print('✓ Warmup successful')
except Exception as e:
    print(f'❌ Warmup failed: {e}')
    raise

# Memory check
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f'✓ GPU Memory after init: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved')

print('\n' + '='*70)
print('✅ SAM 2 MODEL INITIALIZATION COMPLETE')
print('='*70)

Working directory: /Users/kamronaggor/Desktop/School/URI/AI Lab/OceanDetect/sam2-vidio-annotation/models/sam2
Cleared 1 cached modules
⚠ Using CPU
❌ SAM 2 import failed: No module named 'hydra'


ModuleNotFoundError: No module named 'hydra'

## 2: Download Video from Google Drive 

#### We will see the folders that i have in my google drive 

In [ ]:
import subprocess
subprocess.run(["rclone", "lsf", "gdrive:", "--dirs-only", "-R"], check=True)


#### In this case my vidios are in south africa folder

In [ ]:
subprocess.run(["rclone", "lsf", "gdrive:south africa/", "-R"], check=True)


#### Copy the vidio that we are interested in to Unity

In [ ]:
import subprocess
from pathlib import Path

#Download vidio
local_dir = Path("/home/zihara_delgado_uri_edu/videos")
local_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "rclone", 
    "copy", 
    "gdrive:south africa/Copy of (5)GH124317-Etmopterus.mp4", 
    str(local_dir)
], check=True)

print("✓ Video copied to Unity!")



Confirm that the vidio exists

In [ ]:
import os
print(os.path.exists('/home/zihara_delgado_uri_edu/videos/Copy of (5)GH124317-Etmopterus.mp4'))


## 3: Extract and Resize Frames from Video

#### Extracts frames at reduced resolution to fit in GPU memory

In [ ]:
# ============================================
# EXTRACT AND RESIZE FRAMES
# ============================================
# This cell:
# - Extracts frames from video at specified frame rate
# - Resizes frames to 1024x576 (from 1920x1080) to reduce memory usage
# - Estimated GPU memory needed: ~58 GB (fits in H100's 85 GB)

import cv2
import os

# Paths
video_path = os.path.expanduser(f'{path.Path.home()}/videos')
output = os.path.expanduser(f'{path.Path.home()}/annotated_frames')
# frames_dir = f'{output}/video_frames'
video_file = os.path.basename(video_path)
video_name = os.path.splitext(video_file)[0]
frames_dir = f'{output}/{video_name}_frames'
os.makedirs(frames_dir, exist_ok=True)

# Open video
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError("OpenCV could not open the video.")

# Get video properties
video_fps = cap.get(cv2.CAP_PROP_FPS)
if video_fps <= 0:
    video_fps = 30

# Frame extraction settings
FRAME_RATE = 30  # Extract 30 frames per second
frame_interval = int(video_fps / FRAME_RATE)

# Target resolution (reduced from 1920x1080 to fit in GPU memory)
TARGET_WIDTH = 1024
TARGET_HEIGHT = 576

print(f"Video FPS: {video_fps}")
print(f"Extracting every {frame_interval}th frame (FRAME_RATE={FRAME_RATE})")
print(f"Target resolution: {TARGET_WIDTH}x{TARGET_HEIGHT}")

# Extract and resize frames
frame_count = 0
saved_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % frame_interval == 0:
        # Resize frame before saving
        resized_frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
        cv2.imwrite(f'{frames_dir}/{saved_count:05d}.jpg', resized_frame)
        saved_count += 1
    
    frame_count += 1

cap.release()

# Calculate memory estimate
megapixels = (TARGET_WIDTH * TARGET_HEIGHT) / 1e6
estimated_gpu = saved_count * megapixels * 0.1

print(f'\nExtracted {saved_count} frames at {TARGET_WIDTH}x{TARGET_HEIGHT}')
print(f'Estimated GPU memory needed: {estimated_gpu:.1f} GB')

# Count frames for verification
frames = sorted(os.listdir(frames_dir))
num_frames = len(frames)
print(f"Ready for SAM2: {num_frames} frames found")


In [ ]:
import os

# Get video file name without extension
video_file = os.path.basename(video_path)
video_name, _ = os.path.splitext(video_file)

# Make sure output directory exists
frames_dir = f"{output}/video_frames/{video_name}"
os.makedirs(frames_dir, exist_ok=True)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % frame_interval == 0:
        # Resize frame before saving
        resized_frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
        cv2.imwrite(f'{frames_dir}/{saved_count:05d}.jpg', resized_frame)
        saved_count += 1
    
    frame_count += 1

##  4: Preview First 40 Frames (Optional)

preview the first 40 frames, so we take the one where the shark is more clear

In [ ]:
# ============================================
# PREVIEW FIRST 40 FRAMES
# Optional: Visual verification of extracted frames
# ============================================

import cv2
import os
import matplotlib.pyplot as plt

frames_dir = '/home/zihara_delgado_uri_edu/annotated_frames/video_frames'

# Get first 40 frames
frames = sorted(os.listdir(frames_dir))[:40]

plt.figure(figsize=(25, 40))

for i, frame_name in enumerate(frames):
    frame_path = os.path.join(frames_dir, frame_name)
    img = cv2.imread(frame_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(8, 5, i+1)  # 8 rows × 5 columns
    plt.imshow(img)
    plt.title(frame_name, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Displayed first {len(frames)} frames")

##  5: Export Frame for Roboflow Annotation

In [ ]:
# ============================================
# EXPORT FRAME FOR ROBOFLOW ANNOTATION
# ============================================
# This cell:
# - Exports one frame for you to annotate in Roboflow
# - Pick the frame number from the preview above
# - Download this image and upload to Roboflow
# - Use SAM tool in Roboflow for best results

import cv2
import os

frames_dir = '/home/zihara_delgado_uri_edu/annotated_frames/video_frames'

# CHANGE THIS to the frame you want to annotate
# Look at the preview above and pick a clear frame
frame_to_annotate = 23  # ← UPDATE THIS NUMBER

frame_path = os.path.join(frames_dir, f'{frame_to_annotate:05d}.jpg')
output_path = '/home/zihara_delgado_uri_edu/frame_for_roboflow.jpg'

# Copy frame for download
frame = cv2.imread(frame_path)
cv2.imwrite(output_path, frame)

print(f"✓ Exported frame {frame_to_annotate} → frame_for_roboflow.jpg")
print(f"✓ File location: {output_path}")
print(f"\n{'='*70}")
print(f"NEXT STEPS:")
print(f"{'='*70}")
print(f"1. Right-click 'frame_for_roboflow.jpg' in file browser → Download")
print(f"2. Upload to Roboflow")
print(f"3. Annotate the shark using Smart Polygon/SAM tool (NOT manual brush)")
print(f"4. Export as COCO format")
print(f"5. Download the zip file")
print(f"6. Upload zip to Unity")
print(f"7. Continue with Cell 6 below")
print(f"{'='*70}")

# Show the frame
import matplotlib.pyplot as plt
img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.title(f'Frame {frame_to_annotate} - Annotate this in Roboflow', fontsize=14, weight='bold')
plt.axis('off')
plt.show()

# Save frame_index for later use
frame_index = frame_to_annotate
print(f"\n✓ Saved: frame_index = {frame_index}")

## 6: Extract Roboflow Annotation Zip

#### After annotating in Roboflow, upload the zip 

In [ ]:
# ============================================
# EXTRACT ROBOFLOW ANNOTATION
# Run this AFTER uploading your Roboflow COCO export zip
# ============================================

import zipfile
import os

# List available zip files
print("Available zip files:")
for f in os.listdir('/home/zihara_delgado_uri_edu/'):
    if f.endswith('.zip'):
        print(f'  📦 {f}')

# UPDATE THIS to your uploaded zip file name
zip_name = '/home/zihara_delgado_uri_edu/Etmopterus-Uploaded on 02-23-26 at 12-26 pm.coco.zip'

# Extract
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/home/zihara_delgado_uri_edu/roboflow_export')

print('\n✓ Extracted! Contents:')
!find /home/zihara_delgado_uri_edu/roboflow_export -type f

print("\n✓ Ready to load annotation in next cell")

## 7: Load Roboflow Annotation

#### Loads your mask/polygon annotation from Roboflow COCO file.

In [ ]:
# ============================================
# LOAD ROBOFLOW ANNOTATION
# ============================================
# This cell:
# - Loads COCO annotation from Roboflow
# - Handles both polygon and mask (RLE) formats
# - Creates binary mask for SAM 2

import json
import numpy as np
import cv2

# Import pycocotools for mask decoding (if using mask tool in Roboflow)
try:
    from pycocotools import mask as mask_utils
    HAS_PYCOCOTOOLS = True
except ImportError:
    HAS_PYCOCOTOOLS = False

# Load COCO annotation
with open('/home/zihara_delgado_uri_edu/roboflow_export/train/_annotations.coco.json', 'r') as f:
    coco = json.load(f)

roboflow_image = coco['images'][0]
annotation = coco['annotations'][0]
segmentation = annotation['segmentation']

print(f'✓ Roboflow image: {roboflow_image["width"]}x{roboflow_image["height"]}')

# Decode annotation based on format
if isinstance(segmentation, dict):
    # RLE mask format (you used mask/brush tool in Roboflow)
    print('✓ Detected RLE mask format (SAM tool or brush)')
    
    if not HAS_PYCOCOTOOLS:
        print('Installing pycocotools...')
        import subprocess
        subprocess.run(['pip', 'install', 'pycocotools', '--break-system-packages'], check=True)
        from pycocotools import mask as mask_utils
    
    # Decode RLE to binary mask
    mask = mask_utils.decode(segmentation)
    print(f'✓ Decoded mask: {mask.shape}, {np.sum(mask)} pixels')
    
elif isinstance(segmentation, list):
    # Polygon format (you used polygon tool in Roboflow)
    print('✓ Detected polygon format')
    
    polygon = segmentation[0]
    print(f'✓ Found polygon with {len(polygon)//2} points')
    
    # Convert polygon to mask
    height = roboflow_image['height']
    width = roboflow_image['width']
    polygon_np = np.array(polygon).reshape(-1, 2).astype(np.int32)
    
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_np], 1)
    print(f'✓ Created mask from polygon')

else:
    raise ValueError(f"Unknown segmentation format: {type(segmentation)}")

print(f'✓ Using frame {frame_index} (from Cell 5)')
print(f'✓ Mask coverage: {np.sum(mask)} pixels ({100*np.sum(mask)/(mask.shape[0]*mask.shape[1]):.1f}% of image)')

## 8: Visualize Roboflow Annotation

#### Verify annotation matches the extracted frame.

In [ ]:
# ============================================
# VISUALIZE ROBOFLOW ANNOTATION
# ============================================
# Visual check that annotation aligns correctly with extracted frame

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Load the frame
first_frame = Image.open(f'{frames_dir}/{frame_index:05d}.jpg')
width, height = first_frame.size

print(f'✓ Frame {frame_index} size: {width}x{height}')
print(f'✓ Roboflow annotation size: {roboflow_image["width"]}x{roboflow_image["height"]}')

# Check if we need to resize the mask
if width != roboflow_image["width"] or height != roboflow_image["height"]:
    print(f'⚠️ Size mismatch detected - resizing mask to match frame')
    
    scale_x = width / roboflow_image["width"]
    scale_y = height / roboflow_image["height"]
    
    print(f'✓ Scaling by {scale_x:.3f}x (width) and {scale_y:.3f}x (height)')
    
    # Resize mask to match frames
    mask_resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    mask = mask_resized
    
    print(f'✓ Mask resized to {mask.shape}')
else:
    print('✓ Sizes match perfectly!')

# Visualize
frame_rgb = cv2.cvtColor(cv2.imread(f'{frames_dir}/{frame_index:05d}.jpg'), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Original frame
axes[0].imshow(frame_rgb)
axes[0].set_title(f'Frame {frame_index} - Original', fontsize=14, weight='bold')
axes[0].axis('off')

# Mask only
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Roboflow Annotation Mask', fontsize=14, weight='bold')
axes[1].axis('off')

# Overlay
overlay = frame_rgb.copy()
overlay[mask > 0] = overlay[mask > 0] * 0.5 + np.array([255, 0, 0]) * 0.5
axes[2].imshow(overlay)
axes[2].set_title('Frame + Annotation Overlay', fontsize=14, weight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f'\n✅ Mask ready for SAM 2!')
print(f'   Mask shape: {mask.shape}')
print(f'   Mask area: {np.sum(mask)} pixels')
print(f'\nIf the overlay looks correct, proceed to Cell 9')
print(f'If not, check that you annotated frame {frame_index}')

## 9: Initialize SAM 2 with Video Frames

#### Load extracted frames into SAM 2 for tracking.

In [ ]:
# ============================================
# INITIALIZE SAM 2 WITH VIDEO FRAMES
# ============================================
# This cell:
# - Loads all extracted frames into SAM 2
# - Uses CPU offloading to reduce GPU memory usage
# - Creates inference_state for tracking

import torch
import gc

# Clear GPU cache before processing
with torch.cuda.device(device):
    torch.cuda.empty_cache()

# Initialize SAM 2 with frames
inference_state = predictor.init_state(
    video_path=frames_dir,
    offload_video_to_cpu=True,     # Keep frames on CPU to save GPU memory
    offload_state_to_cpu=True      # Offload hidden states to CPU
)

print(f"✓ SAM 2 loaded {saved_count} frames from {frames_dir}")

## 10: Add Initial Annotation to SAM 2

In [ ]:
# ============================================
# ADD INITIAL ANNOTATION TO SAM 2
# ============================================
# This gives SAM 2 your Roboflow annotation as the starting point
# SAM 2 will use this to track the shark through the entire video

# Add the Roboflow mask to SAM 2
_, out_obj_ids, out_mask_logits = predictor.add_new_mask(
    inference_state=inference_state,
    frame_idx=frame_index,  # Frame you annotated in Roboflow
    obj_id=1,               # Object ID for the shark
    mask=mask,              # Your Roboflow mask
)

print(f'✓ SAM 2 initialized with Roboflow annotation on frame {frame_index}!')
print(f'✓ Ready to propagate through {saved_count} frames')

## 11: Define Helper Function

#### Function to remove small disconnected regions (like bite marks).

In [ ]:
# ============================================
# DEFINE MASK CLEANING FUNCTION
# ============================================
# This function removes small disconnected regions from masks
# Keeps only the largest connected component (the shark body)
# Helps remove bite marks and other small artifacts

import cv2
import numpy as np

def clean_mask(mask, min_area=500):
    """
    Remove small disconnected regions from mask.
    Keeps only the largest connected component (shark body).
    
    Args:
        mask: Binary mask (numpy array)
        min_area: Minimum area threshold (unused, kept for compatibility)
    
    Returns:
        Cleaned binary mask
    """
    mask_uint8 = (mask * 255).astype(np.uint8)
    
    # Find all connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask_uint8, 
        connectivity=8
    )
    
    # Keep only the largest component (excluding background label 0)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        cleaned = (labels == largest_label).astype(np.uint8)
    else:
        cleaned = mask_uint8
    
    return cleaned.astype(bool)

print("✓ clean_mask() function defined")

## 12: Propagate Through Video

#### Track the shark through all frames.

In [ ]:
# ============================================
# PROPAGATE THROUGH ALL FRAMES
# ============================================
# This cell:
# - Tracks the shark from your initial annotation through entire video
# - Uses threshold 0.2 (lower = more permissive, catches shark in difficult frames)
# - Applies clean_mask() to remove bite marks and artifacts
# - Clears GPU cache every 100 frames to prevent memory crashes
# - Shows progress and memory usage

import torch
import gc

print('Now propagating through all frames...')
print(f'Total frames to process: {saved_count}')
print(f'Starting from frame {frame_index} (your annotation)')
print(f'Threshold: 0.2 (lower = more permissive)')

video_segments = {}
batch_size = 100  # Clear GPU cache every 100 frames

frame_count = 0

try:
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        # Process mask with threshold and cleaning
        video_segments[out_frame_idx] = {
            out_obj_id: clean_mask((out_mask_logits[i] > 0.2).cpu().numpy().squeeze())
            for i, out_obj_id in enumerate(out_obj_ids)
        }
        
        frame_count += 1
        
        # Clear GPU cache periodically to prevent memory buildup
        if frame_count % batch_size == 0:
            torch.cuda.empty_cache()
            gc.collect()
            
            gpu_mem = torch.cuda.memory_allocated(0) / 1e9
            print(f'✓ Processed {frame_count}/{saved_count} frames | GPU: {gpu_mem:.1f} GB')
        
        # Show progress every 50 frames
        elif out_frame_idx % 50 == 0:
            print(f'  Frame {out_frame_idx}/{saved_count}')

except Exception as e:
    print(f"\n❌ Propagation failed at frame {frame_count}")
    print(f"Error: {e}")
    print(f"\n⚠️ Saved {len(video_segments)} frames processed before crash...")
    
finally:
    # Final cleanup
    torch.cuda.empty_cache()
    gc.collect()
    
    # Print results
    success_rate = 100 * len(video_segments) / saved_count
    
    print(f'\n{"="*70}')
    print(f'✅ PROPAGATION COMPLETE')
    print(f'{"="*70}')
    print(f'Frames tracked: {len(video_segments)}/{saved_count}')
    print(f'Success rate: {success_rate:.1f}%')
    
    if success_rate < 80:
        print(f'\n⚠️ Warning: Success rate < 80%')
        print(f'Consider:')
        print(f'  - Lowering threshold (try 0.15)')
        print(f'  - Adding correction points where tracking failed')
        print(f'  - Using a different starting frame')
    elif success_rate < 95:
        print(f'\n✓ Good tracking with minor gaps')
        print(f'Consider adding correction points for lost frames')
    else:
        print(f'\n✅ Excellent tracking!')
    
    print(f'{"="*70}')

## 14: Export to COCO Format

In [ ]:
# ============================================
# EXPORT TO COCO FORMAT
# ============================================
# This cell:
# - Exports all frames with their annotations
# - Converts masks to polygons (required by Roboflow)
# - Creates COCO JSON file for import to Roboflow
# - Output: /home/zihara_delgado_uri_edu/export_to_roboflow/

import json
import shutil
import os
import cv2
import numpy as np

EXPORT_DIR = '/home/zihara_delgado_uri_edu/export_to_roboflow'
os.makedirs(f'{EXPORT_DIR}/images', exist_ok=True)

# COCO format structure
coco_output = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "shark"}]
}

annotation_id = 1

def mask_to_polygon(mask):
    """Convert binary mask to polygon coordinates."""
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), 
        cv2.RETR_EXTERNAL, 
        cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None
    
    # Get largest contour
    largest = max(contours, key=cv2.contourArea)
    
    # Approximate polygon
    epsilon = 0.002 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)
    
    return approx.flatten().tolist()

print('Exporting annotations...')
print(f'Total frames: {saved_count}')
print(f'Tracked frames: {len(video_segments)}')

for frame_idx in range(saved_count):
    src = f'{frames_dir}/{frame_idx:05d}.jpg'
    dst = f'{EXPORT_DIR}/images/{frame_idx:05d}.jpg'
    
    # Copy frame (copy all frames, even if no annotation)
    shutil.copy(src, dst)
    
    # Add image entry to COCO
    img = cv2.imread(src)
    h, w = img.shape[:2]
    
    coco_output["images"].append({
        "id": frame_idx,
        "file_name": f'{frame_idx:05d}.jpg',
        "width": w,
        "height": h
    })
    
    # Add annotation if SAM 2 tracked the shark in this frame
    if frame_idx not in video_segments or 1 not in video_segments.get(frame_idx, {}):
        continue
    
    mask = video_segments[frame_idx][1]
    polygon = mask_to_polygon(mask)
    
    # Skip if polygon is invalid
    if polygon is None or len(polygon) < 6:
        continue
    
    # Calculate bounding box
    x_coords = polygon[0::2]
    y_coords = polygon[1::2]
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    
    # Add annotation
    coco_output["annotations"].append({
        "id": annotation_id,
        "image_id": frame_idx,
        "category_id": 1,
        "segmentation": [polygon],
        "bbox": [x_min, y_min, x_max - x_min, y_max - y_min],
        "area": cv2.contourArea(np.array(polygon).reshape(-1, 2)),
        "iscrowd": 0
    })
    annotation_id += 1
    
    # Progress update
    if frame_idx % 100 == 0:
        print(f'  Exported {frame_idx}/{saved_count} frames...')

# Save COCO JSON
with open(f'{EXPORT_DIR}/_annotations.coco.json', 'w') as f:
    json.dump(coco_output, f)

print(f'\n{"="*70}')
print(f'✅ EXPORT COMPLETE')
print(f'{"="*70}')
print(f'Total images exported: {len(coco_output["images"])}')
print(f'Total annotations exported: {len(coco_output["annotations"])}')
print(f'Export directory: {EXPORT_DIR}')
print(f'\nNext steps:')
print(f'1. Download the export folder')
print(f'2. Upload to Roboflow as COCO format')
print(f'3. Train your YOLO model!')
print(f'{"="*70}')

## 15: Create zip for download

In [ ]:
# ============================================
# CREATE ZIP FOR DOWNLOAD
# Optional: Compress export folder for easier download
# ============================================

import shutil
import os

zip_filename = '/home/zihara_delgado_uri_edu/shark_annotations_export'
export_dir = '/home/zihara_delgado_uri_edu/export_to_roboflow'

print('Creating zip file...')
shutil.make_archive(zip_filename, 'zip', export_dir)

zip_path = f'{zip_filename}.zip'
zip_size = os.path.getsize(zip_path) / (1024 * 1024)  # Size in MB

print(f'✓ Created: {zip_path}')
print(f'✓ Size: {zip_size:.1f} MB')
print(f'\nRight-click the file in the file browser and select Download')

## 16: Create Comparison Video (Original vs Tracking)

In [ ]:
# ============================================
# VIDEO COMPARISON VISUALIZATION
# Creates side-by-side video: Original vs Annotated
# ============================================
# This cell:
# - Creates a comparison video showing original frames next to SAM 2 annotations
# - Helps visualize tracking quality across the entire video
# - Output: comparison_video.mp4

import cv2
import numpy as np
import os

print("Creating comparison video...")

# Video settings
output_path = '/home/zihara_delgado_uri_edu/comparison_video.mp4'
fps = 5  # Match the frame rate we extracted at

# Get video dimensions from first frame
sample_frame = cv2.imread(f'{frames_dir}/00000.jpg')
frame_height, frame_width = sample_frame.shape[:2]

# Output video will be double width (original + annotated side by side)
output_width = frame_width * 2
output_height = frame_height

# Initialize video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (output_width, output_height))

# Process each frame
frames_processed = 0
frames_with_annotations = 0

for frame_idx in range(saved_count):
    # Load original frame
    frame_path = f'{frames_dir}/{frame_idx:05d}.jpg'
    frame = cv2.imread(frame_path)
    
    if frame is None:
        print(f"⚠️ Could not load frame {frame_idx}")
        continue
    
    # Create annotated version
    annotated_frame = frame.copy()
    
    # Check if we have annotation for this frame
    has_annotation = frame_idx in video_segments and 1 in video_segments.get(frame_idx, {})
    
    if has_annotation:
        mask = video_segments[frame_idx][1]
        
        # Create red overlay on shark
        annotated_frame[mask] = (annotated_frame[mask] * 0.6).astype(np.uint8) + np.array([0, 0, 150], dtype=np.uint8)
        
        # Add green border around mask
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(annotated_frame, contours, -1, (0, 255, 0), 2)
        
        frames_with_annotations += 1
        status_text = "TRACKED"
        status_color = (0, 255, 0)  # Green
    else:
        status_text = "NOT TRACKED"
        status_color = (0, 0, 255)  # Red
    
    # Add text labels
    cv2.putText(frame, "Original", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(annotated_frame, f"SAM 2: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
    
    # Add frame number
    cv2.putText(frame, f"Frame {frame_idx}", (10, frame_height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(annotated_frame, f"Frame {frame_idx}", (10, frame_height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    
    # Combine frames side by side
    combined_frame = np.hstack([frame, annotated_frame])
    
    # Write to video
    out.write(combined_frame)
    
    frames_processed += 1
    
    # Progress update
    if frame_idx % 50 == 0:
        print(f"  Processed {frame_idx}/{saved_count} frames...")

# Release video writer
out.release()

# Get file size
video_size = os.path.getsize(output_path) / (1024 * 1024)  # MB

print(f'\n{"="*70}')
print(f'✅ COMPARISON VIDEO CREATED')
print(f'{"="*70}')
print(f'Output: {output_path}')
print(f'Size: {video_size:.1f} MB')
print(f'Resolution: {output_width}x{output_height}')
print(f'FPS: {fps}')
print(f'Frames processed: {frames_processed}')
print(f'Frames with annotations: {frames_with_annotations} ({100*frames_with_annotations/frames_processed:.1f}%)')
print(f'\nRight-click comparison_video.mp4 in file browser to download')
print(f'{"="*70}')